In [46]:
import plotly
import numpy as np
import plotly.graph_objs as go

In [180]:
import sympy as sp

u, v = sp.symbols("u v")

# is the surface developable
theta = sp.pi/3
A = 0.5
phase = sp.pi
s = u * sp.cos(theta) + v * sp.sin(theta)
gamma = sp.Matrix([u, v, A*sp.cos(s + phase)])

E = (gamma.diff(u)).dot(gamma.diff(u))
F = gamma.diff(u).dot(gamma.diff(v))
G = (gamma.diff(v)).dot(gamma.diff(v))
L = gamma.diff(u, u)
M = gamma.diff(u, v)
N = gamma.diff(v, v)

n = gamma.diff(u).cross(gamma.diff(v))
a = sp.sqrt(n.dot(n))
n = n/a

L = L.dot(n)
M = M.dot(n)
N = N.dot(n)

print((L * N - M**2) / (E * G - F**2))

# if it is zero, then the surface is developable

0


In [181]:
# consider a fully actuated sheet (i.e., all the joints are controllable)
import pinocchio as pin

model, constraints = pin.buildModelFromSdf("mass_mesh_open_tree.sdf", root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("spring_dampers_open_tree.sdf", root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("mass_mesh_open_tree.sdf")
data = model.createData()

In [182]:
# Store all the frame IDs for a grid of masses in a dictionary
# stores the IDs for masses named mass_x{i}_y{j}

frame_ids = dict()

num_elements_x = 5
num_elements_y = 5


for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

In [183]:
q = pin.neutral(model)
# q = pin.randomConfiguration(model)
pin.forwardKinematics(model, data, q)
link_positions = []
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        pin.updateFramePlacement(model, data, frame_ids[(i, j)])
        # print(f"Frame mass_x{i}_y{j} placement:\n{data.oMf[frame_ids[(i, j)]].translation}\n")
        link_positions.append(data.oMf[frame_ids[(i, j)]].translation)

In [242]:
side_length = 0.8
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)

goals_u = np.linspace(u_min, u_max, num_elements_x)
goals_v = np.linspace(v_min, v_max, num_elements_y)

U, V = np.meshgrid(u, v)
goals_U, goals_V = np.meshgrid(goals_u, goals_v)

A = 0.05
angle = np.pi/2
phase = np.pi
frequency = np.pi/0.8


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.5, base_pos=np.array([0, 0, 0])):
    x = u
    y = v
    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.sin(frequency*s + phase) # bump

    # TODO modify x, y, and z such that (0, 0) lies at the base_pos
    offset = base_pos - np.array([0, 0, A * np.sin(frequency*(0) + phase)])
    #                                                          s = 0 at (0,0)
    x += offset[0]
    y += offset[1]
    z += offset[2]

    return x, y, z


goal_positions = dict()



X, Y, Z = gamma_sur(U, V, A=A)
goals_X, goals_Y, goals_Z = gamma_sur(goals_U, goals_V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z),
    # go.Scatter3d(x=goals_X.flatten(), y=goals_Y.flatten(), z=goals_Z.flatten(), 
            # mode='markers', marker=dict(size=23, color='red')),
    go.Scatter3d(x=[pos[0] for pos in link_positions],y=[pos[1] for pos in link_positions],z=[pos[2] for pos in link_positions],
            mode='markers', marker=dict(size=12, color='blue')),
])

fig.update_layout(
    title="γ(u, v): bump surface",
    scene=dict(
        aspectmode='data',
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)


Opening in existing browser session.


In [185]:
# constraints need to be formulated from WE and EW
# we will create a list of frames that need to be linked via constraints
constraint_frames = []
for i in range(-int(num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(-int(num_elements_y/2), int(num_elements_y/2)):
        if i != 0:
            constraint_frames.append([(i, j), (i, j+1)])

len(constraint_frames)

# number of constraints = (n-1)^2

16

In [186]:
elem_dist = side_length / (num_elements_x - 1)

def get_cJacobian(model, data, q, frame_id1, frame_id2, dist):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J1 = pin.getFrameJacobian(model, data, frame_id1, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    J2 = pin.getFrameJacobian(model, data, frame_id2, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

    p1 = data.oMf[frame_id1].translation
    p2 = data.oMf[frame_id2].translation

    constraint_fn_der_wrt_FK = 2*(p1 - p2).reshape(1, 3)

    # relative Jacobian
    J_rel = J2 - J1
    J_rel = J_rel[0:3, :]  # only position constraints
    # print(J_rel.shape, constraint_fn_der_wrt_FK.shape)
    J_const = constraint_fn_der_wrt_FK.dot(J_rel)
    return J_const

def get_cJacobian_for_constraint(model, data, q, constraint_pair, dist):
    frame_id1 = frame_ids[constraint_pair[0]]
    frame_id2 = frame_ids[constraint_pair[1]]
    return get_cJacobian(model, data, q, frame_id1, frame_id2, dist)

def get_cJacobian_matrix(model, data, q, constraint_frames, dist):
    J_c_list = []
    for constraint_pair in constraint_frames:
        J_c = get_cJacobian_for_constraint(model, data, q, constraint_pair, dist)
        J_c_list.append(J_c)
        # print(np.linalg.det(J_c.dot(J_c.T)))
        # must be non zero for a non-singular constraint
    J_c_matrix = np.vstack(J_c_list)
    return J_c_matrix

In [187]:
Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
Jc.shape # should be ((n-1)^2, model.nv)

(16, 72)

In [200]:
goal_positions = dict()
# goal_positions_other = dict()
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        x, y, z = gamma_sur(i*elem_dist, j*elem_dist, A=A, base_pos=np.array([0, 0, 0]))
        goal_positions[(i, j)] = np.array([x, y, z])
        # goal_positions_other[(i, j)] = np.array([goals_X[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Y[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Z[j + int(num_elements_y/2), i + int(num_elements_x/2)]])

In [189]:
# Now, to come up with task jacobians

def get_task_jacobians(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix

def get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    errors = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
        errors.append(err)
        # print(err.shape)
    error_vec = np.hstack(errors)
    # print(error_vec.shape)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix, error_vec


In [190]:
Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

Jt.shape # should be (n^2 * 3, model.nv)

(75, 72)

In [191]:
# now to make a kkt matrix

def make_kkt_matrix(Jc, Jt, reg=1e-6):
    if Jc.shape[1] != Jt.shape[1]:
        raise ValueError("Jc and Jt must have the same number of columns (variables)")
    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    G = Jt.T.dot(Jt) + reg * np.eye(num_vars)
    H = Jc

    KKT_left = np.vstack([G, H])
    KKT_right = np.vstack([H.T, np.zeros((num_constraints, num_constraints))])

    KKT_matrix = np.hstack([KKT_left, KKT_right])
    return KKT_matrix

In [192]:
KKT = make_kkt_matrix(get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist), get_task_jacobians(model, data, q, goal_positions, frame_ids))

KKT.shape

(88, 88)

In [211]:
# make the IK solver
def solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids, reg=1e-6):
    Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
    Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

    KKT = make_kkt_matrix(Jc, Jt, reg=reg)

    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    # right hand side
    b_top = -Jt.T.dot(errors)
    b_bottom = np.zeros((num_constraints, ))
    b = np.hstack([b_top, b_bottom])

    # solve for delta_q and lambda
    sol = np.linalg.solve(KKT, b)
    delta_q = sol[0:num_vars]
    # lambda_vals = sol[num_vars:]

    return delta_q

In [204]:
import time

In [222]:
goal_positions

{(-2, -2): array([-0.4 , -0.4 ,  0.05]),
 (-2, -1): array([-0.4       , -0.2       ,  0.03535534]),
 (-2, 0): array([-0.4,  0. ,  0. ]),
 (-2, 1): array([-0.4       ,  0.2       , -0.03535534]),
 (-2, 2): array([-0.4 ,  0.4 , -0.05]),
 (-1, -2): array([-0.2 , -0.4 ,  0.05]),
 (-1, -1): array([-0.2       , -0.2       ,  0.03535534]),
 (-1, 0): array([-0.2,  0. ,  0. ]),
 (-1, 1): array([-0.2       ,  0.2       , -0.03535534]),
 (-1, 2): array([-0.2 ,  0.4 , -0.05]),
 (0, -2): array([ 0.  , -0.4 ,  0.05]),
 (0, -1): array([ 0.        , -0.2       ,  0.03535534]),
 (0, 0): array([0., 0., 0.]),
 (0, 1): array([ 0.        ,  0.2       , -0.03535534]),
 (0, 2): array([ 0.  ,  0.4 , -0.05]),
 (1, -2): array([ 0.2 , -0.4 ,  0.05]),
 (1, -1): array([ 0.2       , -0.2       ,  0.03535534]),
 (1, 0): array([0.2, 0. , 0. ]),
 (1, 1): array([ 0.2       ,  0.2       , -0.03535534]),
 (1, 2): array([ 0.2 ,  0.4 , -0.05]),
 (2, -2): array([ 0.4 , -0.4 ,  0.05]),
 (2, -1): array([ 0.4       , -0.2     

In [241]:

start = time.time()
for i in range(5):
    v = solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids)
    q = pin.integrate(model, q, v)

end = time.time()

print(f"The time taken to calculate the IK is {end - start} seconds")

The time taken to calculate the IK is 0.18784809112548828 seconds


In [ ]:
### This is to check that the joint Jacobian and frame Jacobian are the same for a given joint

import pinocchio as pin
import numpy as np

model = pin.buildSampleModelManipulator()
data  = model.createData()

q = pin.neutral(model)
pin.forwardKinematics(model, data, q)
pin.updateFramePlacements(model, data)

jid = 3                       # example joint
fid = model.getFrameId(model.names[jid])   # the frame at this joint

# 1) Pinocchio’s "joint Jacobian"
J_joint = pin.computeJointJacobian(model, data, q, jid)

# 2) Pinocchio’s frame Jacobian for the joint’s frame
J_frame = pin.computeFrameJacobian(
    model, data, q, fid, pin.ReferenceFrame.LOCAL
)

print(np.allclose(J_joint, J_frame))


True


In [13]:
for name in model.names:
    print(name)

universe
shoulder1_joint
shoulder2_joint
shoulder3_joint
elbow_joint
wrist1_joint
wrist2_joint
